# Laboratorium terbuka: pendulum nonlinear dan bidang fase

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 3. Seluruh contoh memakai Python terbuka dan dapat dijalankan secara luring setelah paket pada [requirements.lock](requirements.lock) tersedia. Perhitungan bersifat deterministik: tidak ada bilangan acak, grid waktu dan ruang ditetapkan secara eksplisit, serta pemecah numerik memakai toleransi tetap.

**Lisensi dan asal kode.** Notebook ini ditulis baru untuk edisi Bahasa Indonesia dan dimaksudkan untuk didistribusikan di bawah CC BY-NC-SA 4.0 bersama materi turunannya. Notebook ini tidak mengimpor, menyalin, atau menerjemahkan kode MATLAB maupun PPLANE yang bersifat proprieter.

**Tujuan belajar.** Setelah menjalankan notebook ini, pembaca dapat menghubungkan persamaan gerak, energi, bidang fase, lintasan numerik, gesekan, dan klasifikasi linear titik tetap pendulum nonlinear.

## 1. Dari model fisik ke model tak berdimensi

Untuk sudut $\theta(t)$, panjang pendulum $l$, gravitasi $g$, massa $m$, dan konstanta gesekan $c$, model fisiknya adalah

\[
\frac{d^2\theta}{dt^2}=-\frac{g}{l}\sin\theta-\frac{c}{m}\frac{d\theta}{dt}.
\]

Dengan $t_0=\sqrt{l/g}$, $\tau=t/t_0$, $\omega=d\theta/d\tau$, dan $\alpha=(c/m)\sqrt{l/g}$, sistem tak berdimensinya menjadi

\[
\theta'=\omega,\qquad \omega'=-\sin\theta-\alpha\omega.
\]

Energi tak berdimensi $E(\theta,\omega)=\tfrac12\omega^2-\cos\theta$ memenuhi $E'=-\alpha\omega^2$. Jadi, energi kekal ketika $\alpha=0$ dan tidak meningkat ketika $\alpha>0$.

In [ ]:
import os
import numpy as np
import scipy
from scipy.integrate import cumulative_trapezoid, solve_ivp
import matplotlib
import matplotlib.pyplot as plt

VERSI = {
    'NumPy': np.__version__,
    'SciPy': scipy.__version__,
    'Matplotlib': matplotlib.__version__,
}
assert VERSI == {'NumPy': '2.4.4', 'SciPy': '1.17.1', 'Matplotlib': '3.10.9'}

np.set_printoptions(precision=8, suppress=True)
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

WARNA = {
    'biru': '#2369bd',
    'jingga': '#d97706',
    'hijau': '#16825d',
    'merah': '#c83e4d',
    'ungu': '#7c4dba',
    'abu': '#52606d',
}
FIGURES = []
print('Lingkungan eksekusi tervalidasi:', VERSI, '| mesin render:', matplotlib.get_backend())

In [ ]:
def ruas_kanan(tau, keadaan, alpha):
    theta, omega = keadaan
    return np.array([omega, -np.sin(theta) - alpha * omega], dtype=float)


def energi(theta, omega):
    return 0.5 * np.asarray(omega) ** 2 - np.cos(np.asarray(theta))


def potensial(theta):
    return -np.cos(np.asarray(theta))


def jacobian(theta_tetap, alpha):
    return np.array([[0.0, 1.0], [-np.cos(theta_tetap), -alpha]], dtype=float)


def integrasikan(keadaan_awal, alpha, akhir, jumlah=2001):
    waktu = np.linspace(0.0, float(akhir), int(jumlah))
    solusi = solve_ivp(
        ruas_kanan,
        (0.0, float(akhir)),
        np.asarray(keadaan_awal, dtype=float),
        args=(float(alpha),),
        method='DOP853',
        t_eval=waktu,
        rtol=2e-11,
        atol=2e-13,
        max_step=0.025,
    )
    assert solusi.success, solusi.message
    assert solusi.y.shape == (2, jumlah)
    assert np.all(np.isfinite(solusi.y))
    return solusi


def tambahkan_panah(ax, x, y, warna, fraksi=(0.22, 0.52, 0.80)):
    x = np.asarray(x)
    y = np.asarray(y)
    for f in fraksi:
        i = min(max(int(f * (len(x) - 2)), 0), len(x) - 2)
        dx = x[i + 1] - x[i]
        dy = y[i + 1] - y[i]
        if abs(dx) + abs(dy) > 1e-12:
            ax.annotate(
                '',
                xy=(x[i + 1], y[i + 1]),
                xytext=(x[i], y[i]),
                arrowprops={'arrowstyle': '-|>', 'color': warna, 'lw': 1.4},
            )

## 2. Titik tetap dan klasifikasi linear

Titik tetapnya adalah $(n\pi,0)$. Matriks Jacobi di titik tersebut ialah

\[
J(n\pi,0)=\begin{pmatrix}0&1\\-\cos(n\pi)&-\alpha\end{pmatrix}.
\]

Untuk $n$ genap, persamaan karakteristiknya $\lambda^2+\alpha\lambda+1=0$: pusat ketika $\alpha=0$, spiral stabil ketika $0<\alpha^2<4$, simpul stabil kritis ketika $\alpha^2=4$, dan simpul stabil ketika $\alpha^2>4$. Untuk $n$ ganjil, determinannya $-1$, sehingga kedua nilai eigen riil dan berlainan tanda: titik itu selalu merupakan titik pelana.

In [ ]:
def klasifikasi_titik_tetap(n, alpha, toleransi=1e-12):
    assert isinstance(n, (int, np.integer))
    assert alpha >= 0.0
    nilai_eigen = np.linalg.eigvals(jacobian(n * np.pi, alpha))
    if n % 2:
        jenis = 'titik pelana'
    elif abs(alpha) <= toleransi:
        jenis = 'pusat'
    elif abs(alpha * alpha - 4.0) <= toleransi:
        jenis = 'simpul stabil kritis'
    elif alpha * alpha < 4.0:
        jenis = 'spiral stabil'
    else:
        jenis = 'simpul stabil'
    return jenis, nilai_eigen


kasus_uji = [(0, 0.0), (0, 0.6), (0, 2.0), (0, 3.0), (1, 0.6)]
hasil_jenis = [klasifikasi_titik_tetap(n, a)[0] for n, a in kasus_uji]
assert hasil_jenis == [
    'pusat',
    'spiral stabil',
    'simpul stabil kritis',
    'simpul stabil',
    'titik pelana',
]
eigen_kritis = np.linalg.eigvals(jacobian(0.0, 2.0))
assert np.allclose(eigen_kritis, [-1.0, -1.0], rtol=0.0, atol=1e-12)

for (n, a), jenis in zip(kasus_uji, hasil_jenis):
    eig = klasifikasi_titik_tetap(n, a)[1]
    print(f'n={n:>2d}, alpha={a:>3.1f}: {jenis:>23s}; nilai eigen={eig}')

In [ ]:
theta_garis = np.linspace(-2.0 * np.pi, 2.0 * np.pi, 1201)
theta_bidang = np.linspace(-2.0 * np.pi, 2.0 * np.pi, 601)
omega_bidang = np.linspace(-2.8, 2.8, 401)
THETA, OMEGA = np.meshgrid(theta_bidang, omega_bidang)
ENERGI_GRID = energi(THETA, OMEGA)

aras = [-0.5, 0.4, 1.0, 1.6]
warna_aras = [WARNA['biru'], WARNA['hijau'], WARNA['merah'], WARNA['ungu']]
label_aras = ['E = -0,5 (< 1)', 'E = 0,4 (< 1)', 'E = 1', 'E = 1,6 (> 1)']

fig, (ax_v, ax_f) = plt.subplots(1, 2, figsize=(12.4, 4.8))
ax_v.plot(theta_garis, potensial(theta_garis), color=WARNA['abu'], lw=2.4, label=r'$V(\theta)=-\cos\theta$')
for e, warna, label in zip(aras, warna_aras, label_aras):
    ax_v.axhline(e, color=warna, lw=1.7, ls='--' if e != 1.0 else '-', label=label)
ax_v.set(xlabel=r'$\theta$', ylabel='energi', title='Potensial dan aras energi')
ax_v.set_xlim(theta_garis[0], theta_garis[-1])
ax_v.set_ylim(-1.25, 1.9)
ax_v.legend(loc='upper right', fontsize=8)

kontur = ax_f.contour(THETA, OMEGA, ENERGI_GRID, levels=aras, colors=warna_aras, linewidths=[1.7, 1.7, 2.5, 1.7])
ax_f.clabel(kontur, fmt={-0.5: 'E=-0,5', 0.4: 'E=0,4', 1.0: 'E=1', 1.6: 'E=1,6'}, fontsize=8)
genap = np.array([-2, 0, 2]) * np.pi
ganjil = np.array([-1, 1]) * np.pi
ax_f.scatter(genap, np.zeros_like(genap), s=42, marker='o', color=WARNA['biru'], label='pusat stabil netral')
ax_f.scatter(ganjil, np.zeros_like(ganjil), s=54, marker='x', color=WARNA['merah'], label='titik pelana')
ax_f.set(xlabel=r'$\theta$', ylabel=r'$\omega=d\theta/d\tau$', title=r'Bidang fase konservatif ($\alpha=0$)')
ax_f.set_xlim(theta_bidang[0], theta_bidang[-1])
ax_f.set_ylim(omega_bidang[0], omega_bidang[-1])
ax_f.legend(loc='upper right', fontsize=8)

fig.suptitle('Gambar 1. Potensial dan geometri aras energi', fontsize=13)
fig.tight_layout()
FIGURES.append(fig)
fig.canvas.draw()
fig

### Deskripsi panjang Gambar 1

Gambar terdiri atas dua panel dengan rentang sudut dari $-2\pi$ sampai $2\pi$. Panel kiri menampilkan kurva potensial periodik $V(\theta)=-\cos\theta$, dengan lembah pada kelipatan genap $\pi$ dan puncak pada kelipatan ganjil $\pi$. Empat garis horizontal menunjukkan energi $E=-0{,}5$, $0{,}4$, $1$, dan $1{,}6$. Panel kanan memperlihatkan kontur energi yang sama pada bidang $(\theta,\omega)$. Kontur dengan $-1<E<1$ membentuk orbit tertutup di sekitar pusat berpenanda lingkaran biru; aras $E=-1$ mereduksi menjadi kesetimbangan, kontur $E=1$ berwarna merah membentuk separatriks melalui titik pelana berpenanda silang, dan kontur $E>1$ berwarna ungu membentang melintasi sumur potensial sehingga mewakili rotasi. Susunan dua panel menegaskan bahwa memotong lanskap potensial pada suatu tinggi energi menghasilkan geometri orbit pada bidang fase.

In [ ]:
sol_osilasi = integrasikan((1.4, 0.0), alpha=0.0, akhir=18.0, jumlah=1801)
sol_separatriks = integrasikan((0.0, 2.0), alpha=0.0, akhir=12.0, jumlah=1601)
sol_rotasi = integrasikan((-5.7, 2.4), alpha=0.0, akhir=4.8, jumlah=1201)
sol_redam_1 = integrasikan((-2.2, 0.0), alpha=0.35, akhir=30.0, jumlah=3001)
sol_redam_2 = integrasikan((1.6, 1.1), alpha=0.35, akhir=30.0, jumlah=3001)

theta_q = np.linspace(-2.0 * np.pi, 2.0 * np.pi, 29)
omega_q = np.linspace(-2.8, 2.8, 23)
TQ, OQ = np.meshgrid(theta_q, omega_q)

def gambar_medan(ax, alpha, judul):
    U = OQ
    V = -np.sin(TQ) - alpha * OQ
    panjang = np.hypot(U, V)
    panjang = np.where(panjang > 1e-14, panjang, 1.0)
    ax.quiver(TQ, OQ, U / panjang, V / panjang, color='#a7b1ba', alpha=0.75, pivot='mid', scale=34)
    genap = np.array([-2, 0, 2]) * np.pi
    ganjil = np.array([-1, 1]) * np.pi
    ax.scatter(genap, np.zeros_like(genap), s=42, marker='o', color=WARNA['biru'], zorder=5, label='kesetimbangan genap')
    ax.scatter(ganjil, np.zeros_like(ganjil), s=54, marker='x', color=WARNA['merah'], zorder=5, label='titik pelana')
    ax.set(xlim=(-2.0 * np.pi, 2.0 * np.pi), ylim=(-2.8, 2.8), xlabel=r'$\theta$', ylabel=r'$\omega$', title=judul)


fig, (ax_k, ax_r) = plt.subplots(1, 2, figsize=(12.4, 4.9), sharey=True)
gambar_medan(ax_k, 0.0, r'Tanpa gesekan: $\alpha=0$')
for solusi, warna, label in [
    (sol_osilasi, WARNA['biru'], 'osilasi, -1<E<1'),
    (sol_separatriks, WARNA['merah'], 'separatriks, E=1'),
    (sol_rotasi, WARNA['ungu'], 'rotasi, E>1'),
]:
    ax_k.plot(solusi.y[0], solusi.y[1], color=warna, lw=2.0, label=label)
    tambahkan_panah(ax_k, solusi.y[0], solusi.y[1], warna)
ax_k.legend(loc='upper right', fontsize=8)

gambar_medan(ax_r, 0.35, r'Dengan gesekan: $\alpha=0{,}35$')
for solusi, warna, label in [
    (sol_redam_1, WARNA['jingga'], 'lintasan A'),
    (sol_redam_2, WARNA['hijau'], 'lintasan B'),
]:
    ax_r.plot(solusi.y[0], solusi.y[1], color=warna, lw=2.0, label=label)
    tambahkan_panah(ax_r, solusi.y[0], solusi.y[1], warna, fraksi=(0.06, 0.16, 0.34))
ax_r.legend(loc='upper right', fontsize=8)

fig.suptitle('Gambar 2. Medan vektor, lintasan numerik, dan arah gerak', fontsize=13)
fig.tight_layout()
FIGURES.append(fig)
fig.canvas.draw()
fig

### Deskripsi panjang Gambar 2

Kedua panel memakai sumbu horizontal $\theta$ dari $-2\pi$ sampai $2\pi$ dan sumbu vertikal $\omega$ dari $-2{,}8$ sampai $2{,}8$. Anak panah abu-abu membentuk medan vektor; kepala panah kecil pada kurva berwarna menunjukkan arah waktu. Pada panel kiri tanpa gesekan, orbit biru tertutup menggambarkan osilasi berenergi kurang dari satu, kurva merah mendekati titik pelana dan menandai separatriks berenergi satu, sedangkan lintasan ungu berenergi lebih dari satu bergerak melintasi lebih dari satu sumur potensial. Pada panel kanan dengan $\alpha=0{,}35$, dua lintasan berwarna kehilangan energi dan melingkar ke dalam menuju kesetimbangan genap. Lingkaran biru menandai kesetimbangan pada kelipatan genap $\pi$; silang merah menandai titik pelana pada kelipatan ganjil $\pi$.

In [ ]:
alpha_redam = 0.35
sol_kekal = integrasikan((1.2, 0.4), alpha=0.0, akhir=50.0, jumlah=5001)
sol_redam = integrasikan((0.0, 2.4), alpha=alpha_redam, akhir=50.0, jumlah=5001)

E_kekal = energi(sol_kekal.y[0], sol_kekal.y[1])
E_redam = energi(sol_redam.y[0], sol_redam.y[1])
galat_kekalan = float(np.max(np.abs(E_kekal - E_kekal[0])))
kenaikan_terbesar = float(np.max(np.diff(E_redam)))

disipasi = alpha_redam * cumulative_trapezoid(sol_redam.y[1] ** 2, sol_redam.t, initial=0.0)
perubahan_energi = E_redam - E_redam[0]
galat_neraca = float(np.max(np.abs(perubahan_energi + disipasi)))

assert galat_kekalan < 2e-9
assert kenaikan_terbesar < 2e-10
assert galat_neraca < 3e-5
assert E_redam[-1] < -0.995
assert E_redam[-1] >= -1.0 - 2e-10

fig, (ax_e, ax_n) = plt.subplots(1, 2, figsize=(12.4, 4.5))
ax_e.plot(sol_kekal.t, E_kekal, color=WARNA['biru'], lw=2.0, label=r'$\alpha=0$: energi kekal')
ax_e.plot(sol_redam.t, E_redam, color=WARNA['jingga'], lw=2.0, label=r'$\alpha=0{,}35$: energi menurun')
ax_e.axhline(-1.0, color=WARNA['abu'], ls='--', lw=1.4, label='batas bawah E=-1')
ax_e.set(xlabel=r'$\tau$', ylabel='E', title='Energi sepanjang lintasan')
ax_e.legend(fontsize=8)

ax_n.plot(sol_redam.t, perubahan_energi, color=WARNA['merah'], lw=2.1, label=r'$E(\tau)-E(0)$')
ax_n.plot(sol_redam.t, -disipasi, color=WARNA['ungu'], lw=1.5, ls='--', label=r'$-\alpha\int_0^\tau\omega^2\,ds$')
ax_n.set(xlabel=r'$\tau$', ylabel='perubahan energi', title='Pemeriksaan neraca disipasi')
ax_n.legend(fontsize=8)

fig.suptitle('Gambar 3. Kekekalan dan disipasi energi', fontsize=13)
fig.tight_layout()
FIGURES.append(fig)
fig.canvas.draw()
fig

print(f'Galat maksimum energi konservatif : {galat_kekalan:.3e}')
print(f'Kenaikan langkah terbesar, redaman: {kenaikan_terbesar:.3e}')
print(f'Galat maksimum neraca disipasi     : {galat_neraca:.3e}')

### Deskripsi panjang Gambar 3

Panel kiri membandingkan energi terhadap waktu tak berdimensi $\tau$ dari nol sampai lima puluh. Kurva biru untuk $\alpha=0$ tampak horizontal pada energi awalnya; kurva jingga untuk $\alpha=0{,}35$ turun dari nilai di atas satu dan mendekati batas bawah $E=-1$ yang ditandai garis abu-abu putus-putus. Panel kanan menumpangtindihkan perubahan energi numerik berwarna merah dengan negatif integral disipasi $-\alpha\int_0^\tau\omega^2\,ds$ berwarna ungu putus-putus. Kedua kurva hampir tepat berimpit. Hal ini memeriksa secara numerik identitas $E'=-\alpha\omega^2$ dan menunjukkan bahwa energi sistem teredam tidak meningkat.

In [ ]:
E_osilasi = float(energi(1.4, 0.0))
E_separatriks = float(energi(0.0, 2.0))
E_rotasi = float(energi(-5.7, 2.4))
assert E_osilasi < 1.0
assert np.isclose(E_separatriks, 1.0, rtol=0.0, atol=1e-14)
assert E_rotasi > 1.0

for n in range(-3, 4):
    assert np.allclose(ruas_kanan(0.0, (n * np.pi, 0.0), 0.35), (0.0, 0.0), rtol=0.0, atol=2e-15)

keadaan_uji = np.array([0.73, -1.21])
alpha_uji = 0.47
gradien_E = np.array([np.sin(keadaan_uji[0]), keadaan_uji[1]])
turunan_dari_medan = float(gradien_E @ ruas_kanan(0.0, keadaan_uji, alpha_uji))
turunan_teoretis = float(-alpha_uji * keadaan_uji[1] ** 2)
assert np.isclose(turunan_dari_medan, turunan_teoretis, rtol=2e-15, atol=2e-15)
assert len(FIGURES) == 3

print('Semua pemeriksaan lulus.')
print(f'-1<E<1: {E_osilasi:.6f}; E=1: {E_separatriks:.6f}; E>1: {E_rotasi:.6f}')
print(f'dE/dtau analitik={turunan_teoretis:.12f}; dari medan={turunan_dari_medan:.12f}')